# End-to-End License Plate Recognition (LPR) System
### Deep Learning Capstone Project — Indian Vehicles

## Problem Statement
Manual vehicle identification for tasks like toll collection, traffic monitoring, and law
enforcement is slow and error-prone at scale. This project builds an **end-to-end, real-time
Automatic License Plate Recognition (ALPR) pipeline** for Indian vehicles using a **two-stage
deep learning architecture**:

1. **Detection Stage** — YOLO-based object detector locates the license plate region in an image/frame.
2. **Recognition Stage** — An OCR model (transformer-based) reads the alphanumeric text from the cropped plate.

The system is trained and evaluated on an Indian license plate dataset, optimized for real-time
inference, and evaluated using detection (mAP, IoU) and recognition (character accuracy, plate
accuracy) metrics. Target use cases: **toll collection, traffic monitoring, automated vehicle
identification.**

## Workflow 
1. Vehicle identification need → automated LPR
2. Architecture: end-to-end LPR pipeline — Detection → Recognition
3. DL Models: YOLO (plate detection) + Transformer OCR (text recognition)
4. Dataset: Indian license plate dataset (~8-9k images)
5. Data preprocessing & character segmentation
6. Two-stage architecture (Detection model + Recognition model, trained separately)
7. Training & optimization
8. Real-time LPR performance metrics & accuracy
9. Deployment considerations: traffic monitoring, toll collection


## 1. Environment Setup
Install the required libraries: `ultralytics` (YOLOv8) for detection, `easyocr` and `transformers` (TrOCR) for recognition, plus standard CV/DS tooling.

In [ ]:
# Run once
!pip install ultralytics easyocr transformers opencv-python-headless pillow matplotlib pandas scikit-learn -q


In [ ]:
import os
import cv2
import glob
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
print("CUDA available:", torch.cuda.is_available())


## 2. Dataset

Use an **Indian license plate dataset** (~8-9k annotated images). Good public sources:
- Kaggle: "Indian Number Plates" / "Indian Vehicle Dataset"
- OpenALPR benchmark dataset (for cross-validation on plate formats)

**Expected folder structure (YOLO format):**
```
data/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/    # YOLO .txt: class x_center y_center width height (normalized)
    ├── val/
    └── test/
```

Download your dataset and place it under `data/` following this structure before running the
training cells below. A `data.yaml` config is generated automatically in the next cell.


In [ ]:
DATA_DIR = Path("data")
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

for split in ["train", "val", "test"]:
    (IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (LABELS_DIR / split).mkdir(parents=True, exist_ok=True)

data_yaml = f"""
path: {DATA_DIR.resolve()}
train: images/train
val: images/val
test: images/test

names:
  0: license_plate
"""

with open(DATA_DIR / "data.yaml", "w") as f:
    f.write(data_yaml)

print(data_yaml)
print("Train images found:", len(list((IMAGES_DIR / 'train').glob('*'))))


In [ ]:
# Quick sanity check: visualize a few images with their YOLO bounding boxes
def show_sample(split="train", n=4):
    img_paths = list((IMAGES_DIR / split).glob("*"))[:n]
    if not img_paths:
        print(f"No images found in {split} split yet — add your dataset first.")
        return
    fig, axes = plt.subplots(1, len(img_paths), figsize=(4*len(img_paths), 4))
    if len(img_paths) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, img_paths):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = LABELS_DIR / split / (img_path.stem + ".txt")
        if label_path.exists():
            for line in open(label_path):
                cls, xc, yc, bw, bh = map(float, line.split())
                x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        ax.imshow(img); ax.axis("off")
    plt.tight_layout(); plt.show()

show_sample()


## 3. Data Preprocessing & Character Segmentation

Before/alongside OCR, classical preprocessing improves recognition robustness:
- Grayscale conversion
- Contrast enhancement (CLAHE)
- Adaptive thresholding
- Contour-based character segmentation (useful as a fallback / for analysis even though the
  transformer OCR model reads the full plate crop directly).


In [ ]:
def preprocess_plate(plate_img):
    """Grayscale -> CLAHE contrast enhancement -> adaptive threshold."""
    gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    thresh = cv2.adaptiveThreshold(
        enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 8
    )
    return gray, enhanced, thresh


def segment_characters(thresh_img, min_area=50):
    """Contour-based character segmentation (classical fallback / diagnostics)."""
    contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        area = w * h
        aspect_ratio = h / float(w + 1e-6)
        if area > min_area and 1.0 < aspect_ratio < 6.0:
            boxes.append((x, y, w, h))
    boxes = sorted(boxes, key=lambda b: b[0])  # left-to-right order
    return boxes


def visualize_preprocessing(plate_img):
    gray, enhanced, thresh = preprocess_plate(plate_img)
    boxes = segment_characters(thresh)
    seg_vis = cv2.cvtColor(thresh.copy(), cv2.COLOR_GRAY2BGR)
    for (x, y, w, h) in boxes:
        cv2.rectangle(seg_vis, (x, y), (x + w, y + h), (0, 255, 0), 1)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, img, title in zip(
        axes, [plate_img, gray, thresh, seg_vis],
        ["Original crop", "Grayscale + CLAHE", "Adaptive threshold", f"Segmented chars ({len(boxes)})"]
    ):
        if img.ndim == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap="gray")
        ax.set_title(title); ax.axis("off")
    plt.tight_layout(); plt.show()
    return boxes

# Example usage once you have a cropped plate image:
# plate_crop = cv2.imread("data/sample_plate_crop.jpg")
# visualize_preprocessing(plate_crop)


## 4. Stage 1 — License Plate Detection (YOLOv8)

Train a YOLOv8 model to localize license plates. This is the **detection** half of the two-stage
architecture.


In [ ]:
from ultralytics import YOLO

# Start from a pretrained COCO checkpoint and fine-tune on the plate dataset (transfer learning)
detector = YOLO("yolov8n.pt")  # 'n' = nano, fastest; use 'yolov8s.pt' for better accuracy

results = detector.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,          # early stopping
    optimizer="AdamW",
    lr0=1e-3,
    project="runs/detect",
    name="lpr_yolov8",
)


In [ ]:
# Validate the trained detector
metrics = detector.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)


In [ ]:
BEST_DETECTOR_WEIGHTS = "runs/detect/lpr_yolov8/weights/best.pt"
detector = YOLO(BEST_DETECTOR_WEIGHTS)

def detect_plates(image_path, conf=0.4):
    """Run detector on an image, return list of (x1,y1,x2,y2,conf) plate boxes."""
    results = detector.predict(image_path, conf=conf, verbose=False)[0]
    boxes = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        boxes.append((x1, y1, x2, y2, float(box.conf[0])))
    return boxes


## 5. Stage 2 — Plate Text Recognition (Transformer OCR)

Two OCR options are wired up — pick whichever performs better on your data:
- **EasyOCR** — fast CRNN-based OCR, good baseline
- **TrOCR (transformer OCR, HuggingFace)** — attention/transformer-based recognition, matches
  your notes and generally handles varied fonts/angles better


In [ ]:
import easyocr
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

def ocr_easyocr(plate_crop):
    result = reader.readtext(plate_crop, detail=0, paragraph=False)
    text = "".join(result).upper().replace(" ", "")
    return text


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
trocr_model.eval()
if torch.cuda.is_available():
    trocr_model.to("cuda")

def ocr_trocr(plate_crop):
    image = Image.fromarray(cv2.cvtColor(plate_crop, cv2.COLOR_BGR2RGB)).convert("RGB")
    pixel_values = trocr_processor(images=image, return_tensors="pt").pixel_values
    if torch.cuda.is_available():
        pixel_values = pixel_values.to("cuda")
    generated_ids = trocr_model.generate(pixel_values, max_new_tokens=16)
    text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text.upper().replace(" ", "")


In [ ]:
import re

INDIAN_PLATE_REGEX = re.compile(r"^[A-Z]{2}[0-9]{1,2}[A-Z]{1,3}[0-9]{4}$")

def clean_and_validate(text):
    """Strip non-alphanumeric chars, uppercase, check against Indian plate format."""
    cleaned = re.sub(r"[^A-Z0-9]", "", text.upper())
    is_valid = bool(INDIAN_PLATE_REGEX.match(cleaned))
    return cleaned, is_valid


## 6. Full Two-Stage Pipeline (Detection → Recognition)

In [ ]:
def recognize_plate(image_path, ocr_engine="trocr", conf=0.4, show=False):
    """End-to-end: detect plate(s) in an image, run OCR on each crop, validate format."""
    image = cv2.imread(image_path)
    boxes = detect_plates(image_path, conf=conf)

    results = []
    for (x1, y1, x2, y2, det_conf) in boxes:
        crop = image[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        raw_text = ocr_trocr(crop) if ocr_engine == "trocr" else ocr_easyocr(crop)
        cleaned, is_valid = clean_and_validate(raw_text)
        results.append({
            "bbox": (x1, y1, x2, y2),
            "det_conf": det_conf,
            "raw_text": raw_text,
            "plate_text": cleaned,
            "valid_format": is_valid,
        })

    if show:
        vis = image.copy()
        for r in results:
            x1, y1, x2, y2 = r["bbox"]
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(vis, r["plate_text"], (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

    return results

# Example:
# recognize_plate("data/images/test/sample.jpg", ocr_engine="trocr", show=True)


## 7. Training & Optimization Notes

- **Augmentation**: YOLOv8's built-in augmentations (mosaic, HSV jitter, flip) help generalize
  across lighting/angle. Disable horizontal flip if plate text orientation matters (it does).
- **Learning rate schedule**: cosine decay with warmup (`ultralytics` handles this via `lr0`/`lrf`).
- **Early stopping**: `patience` parameter above stops training when val mAP plateaus.
- **Model size trade-off**: `yolov8n` (nano) for real-time speed vs `yolov8s/m` for accuracy —
  benchmark both if latency is a hard constraint (e.g. toll booths).
- **OCR fine-tuning**: TrOCR can be fine-tuned on cropped plate images + ground-truth text using
  HuggingFace `Trainer` if pretrained accuracy isn't sufficient for your dataset's fonts.


In [ ]:
# Disable flip augmentation explicitly when re-training (text orientation matters for plates)
# detector.train(data=str(DATA_DIR / "data.yaml"), epochs=50, imgsz=640, fliplr=0.0, flipud=0.0)


## 8. Evaluation Metrics

- **Detection**: mAP@0.5, mAP@0.5:0.95, IoU (from YOLO's built-in validation)
- **Recognition**: Character-level accuracy, full plate (exact match) accuracy
- **End-to-end**: Correct detections + correct OCR / total plates


In [ ]:
def char_accuracy(pred, gt):
    """Character-level accuracy between predicted and ground-truth plate strings."""
    if not gt:
        return 0.0
    matches = sum(1 for p, g in zip(pred, gt) if p == g)
    return matches / max(len(pred), len(gt))


def evaluate_recognition(test_csv):
    """test_csv: columns ['image_path', 'ground_truth_plate']"""
    df = pd.read_csv(test_csv)
    char_accs, exact_matches = [], []
    for _, row in df.iterrows():
        preds = recognize_plate(row["image_path"], ocr_engine="trocr")
        pred_text = preds[0]["plate_text"] if preds else ""
        gt = row["ground_truth_plate"].upper()
        char_accs.append(char_accuracy(pred_text, gt))
        exact_matches.append(int(pred_text == gt))

    print(f"Mean character accuracy: {np.mean(char_accs):.3f}")
    print(f"Exact plate match accuracy: {np.mean(exact_matches):.3f}")
    return df.assign(char_acc=char_accs, exact_match=exact_matches)

# evaluate_recognition("data/test_labels.csv")


## 9. Real-Time Inference (Video) & Deployment Notes

The function below runs the pipeline on video frames and reports FPS — the key metric for
real-time use cases like **toll collection** and **traffic monitoring**. No web frontend is
included here (notebook/CLI only, per project scope) — this function is what a deployment
service would wrap.


In [ ]:
def run_on_video(video_path, ocr_engine="trocr", conf=0.4, frame_skip=2, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    frame_idx = 0
    processed = 0
    detections_log = []
    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_skip == 0:
            tmp_path = "_tmp_frame.jpg"
            cv2.imwrite(tmp_path, frame)
            results = recognize_plate(tmp_path, ocr_engine=ocr_engine, conf=conf)
            for r in results:
                r["frame"] = frame_idx
                detections_log.append(r)
            processed += 1
            if max_frames and processed >= max_frames:
                break
        frame_idx += 1

    cap.release()
    elapsed = time.time() - start_time
    fps = processed / elapsed if elapsed > 0 else 0
    print(f"Processed {processed} frames in {elapsed:.2f}s ({fps:.2f} FPS)")
    return pd.DataFrame(detections_log)

# results_df = run_on_video("data/sample_traffic.mp4", frame_skip=3, max_frames=100)


### Deployment considerations
- **Toll collection**: pair detections with a vehicle-class check (car/truck/bike) and log to a
  billing DB; require format validation (`valid_format`) before charging.
- **Traffic monitoring**: run at reduced frame rate (`frame_skip`) on live camera feeds; log
  plate + timestamp + camera ID for downstream analytics (speed, congestion, violations).
- **Latency budget**: nano YOLO + EasyOCR is fastest; TrOCR is more accurate but slower —
  benchmark both on your target hardware (edge device vs server) before choosing.
- **Privacy**: plate data is PII in most jurisdictions — hash/encrypt at rest, restrict retention.

